**Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.** 

Driver: The Driver is the main program that creates the Spark session, converts the user's code into an execution plan (DAG), schedules tasks, and sends them to the executors. It also collects the results after execution.

Cluster Manager: It manages the available resources in the cluster. It allocates CPU and memory to the Spark application and launches executors on the available worker nodes.

Executor: It is a worker process that performs the actual computation. They execute the tasks assigned by the Driver, process the data, store intermediate results in memory or disk, and send the final results back to the Driver.

**Q2: How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?**

Ans: Lazy Evaluation means whenever we try to perform transformations, these transformations are not executed immediately. Instead, Spark records these operations and creates a Directed Acyclic Graph (DAG). The trnasformations are executed only when an action is called.

So what happens is, this allows spark to optimize the entire execution plan. It combines multiple operations, removes unnecessary computations, and reduces data movement and disk I/O. As a result, large datasets are processed faster and more efficiently.


**Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.** 

In [0]:
df = spark.read.csv("/Volumes/workspace/default/myFiles/sales.csv", header = True,inferSchema = True)
df.show(10)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|transaction_id|user_id|transaction_date|       username|               email|age|subscription|region|     city|store_id|product_category|sale_amount|  price|   status|      raw_timestamp|
+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|           130|   1160|      2026-01-01|  whitneyjustin|derrickhale@examp...| 48|       Basic| South|Hyderabad|       6|     Electronics|    3399.78| 389.57|     NULL|1982-12-26 19:49:53|
|           222|   1121|      2025-08-19|     jonathan70| scott37@example.org| 42|       Basic|  East|  Chennai|      20|           Books|    2110.99|1869.75|  Pending|1993-12-14 00:30:06|
|           227|   1055|      2026-03-29|      dpeterso

**Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?**

CSV stores data row by row. It takes more storage space and is slower to read for large datasets.

Parquet stores data column by column. It uses less storage due to compression and provides faster query performance because Spark reads only the required columns instead of the entire dataset.

**Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.**

In [0]:
df1 = df.filter(df.product_category == "Electronics").select("transaction_id", "price")
df1.show(10)

+--------------+-------+
|transaction_id|  price|
+--------------+-------+
|           130| 389.57|
|           227|2982.99|
|           249| 719.39|
|             8|1001.47|
|            75| 359.58|
|           227|1022.31|
|            47| 930.88|
|           232|1629.51|
|            31|1503.42|
|           122| 375.94|
+--------------+-------+
only showing top 10 rows


**Q6: Write the code to "revise" a DataFrame by renaming the column username to customer_name and casting the price column from a String to a Double.**

In [0]:
from pyspark.sql.functions import col

df1 = df.withColumnRenamed("username", "customer_name").withColumn("price", col("price").cast("double"))
df1.show(10)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|transaction_id|user_id|transaction_date|  customer_name|               email|age|subscription|region|     city|store_id|product_category|sale_amount|  price|   status|      raw_timestamp|
+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|           130|   1160|      2026-01-01|  whitneyjustin|derrickhale@examp...| 48|       Basic| South|Hyderabad|       6|     Electronics|    3399.78| 389.57|     NULL|1982-12-26 19:49:53|
|           222|   1121|      2025-08-19|     jonathan70| scott37@example.org| 42|       Basic|  East|  Chennai|      20|           Books|    2110.99|1869.75|  Pending|1993-12-14 00:30:06|
|           227|   1055|      2026-03-29|      dpeterso

**Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?**

Ans: Spark uses the Lineage Graph (DAG) to keep track of all the transformations performed on the data. If a worker node fails, Spark uses the DAG to recompute only the lost data instead of processing the entire dataset again. This provides fault tolerance.

**Q8: Write a query to filter a DataFrame df for rows where the status is 'Completed' AND the sales amount is greater than 1000.** 

In [0]:
df1 = df.filter((df.status == "Completed") & (df.sale_amount > 1000))
df1.show(10)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|transaction_id|user_id|transaction_date|       username|               email|age|subscription|region|     city|store_id|product_category|sale_amount|  price|   status|      raw_timestamp|
+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|           227|   1055|      2026-03-29|      dpeterson|martinezmichele@e...| 41|     Premium| North|Bangalore|       2|     Electronics|    4754.58|2982.99|Completed|1988-01-24 20:32:51|
|           249|   1051|      2025-10-28|    byrddeborah|maldonadotoni@exa...| 20|       Basic|  West|  Chennai|       9|     Electronics|    3217.93| 719.39|Completed|1973-03-24 00:16:21|
|           151|   1016|      2026-05-30|franklinkristi

**Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.**

Spark applies the filter condition while reading a Parquet file. It loads only the required data instead of reading the whole file. This way it reduces the amount of data being loaded into the memory, this technique is called Predicate Pushdown.

**Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).**

In [0]:
df1 = df.withColumn("final_price", col("price") * 1.18)
df1.show(10)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+------------------+
|transaction_id|user_id|transaction_date|       username|               email|age|subscription|region|     city|store_id|product_category|sale_amount|  price|   status|      raw_timestamp|       final_price|
+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+------------------+
|           130|   1160|      2026-01-01|  whitneyjustin|derrickhale@examp...| 48|       Basic| South|Hyderabad|       6|     Electronics|    3399.78| 389.57|     NULL|1982-12-26 19:49:53|459.69259999999997|
|           222|   1121|      2025-08-19|     jonathan70| scott37@example.org| 42|       Basic|  East|  Chennai|      20|           Books|    2110.99|1869.75|  Pending|

**Q11: What is the difference between Transformations and Actions? Provide two examples of each.**

Transformations are operations that create a new DataFrame but are not executed immediately. Examples are filter() and select().

Actions are operations that execute the transformations and return the result. Examples are show() and collect().


**Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".** 

In [0]:
dfp = spark.read.parquet("path/to/input")
dfp.filter("user_id IS NOT NULL").write.csv("path/to/output")

**Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?**

In Client Mode, the Driver runs on the local machine where the application is started.

In Cluster Mode, the Driver runs inside the cluster on a worker node, so the application continues even if the client disconnects.

**Q14: Write a query to filter the DataFrame df for rows where the region is 'North' OR the status is 'Completed'**

In [0]:
df1 = df.filter((df.region == "North") | (df.status == "Completed"))
df1.show(10)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|transaction_id|user_id|transaction_date|       username|               email|age|subscription|region|     city|store_id|product_category|sale_amount|  price|   status|      raw_timestamp|
+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|           227|   1055|      2026-03-29|      dpeterson|martinezmichele@e...| 41|     Premium| North|Bangalore|       2|     Electronics|    4754.58|2982.99|Completed|1988-01-24 20:32:51|
|           249|   1051|      2025-10-28|    byrddeborah|maldonadotoni@exa...| 20|       Basic|  West|  Chennai|       9|     Electronics|    3217.93| 719.39|Completed|1973-03-24 00:16:21|
|           151|   1016|      2026-05-30|franklinkristi

**Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?**

.show(5) displays only the first 5 rows, so it uses less memory, .collect() loads the entire dataset into memory, which can cause memory issues for very large datasets.